# Lab 2 · Hyperparameter tuning with nested runs
**Block 1 · about 30 min.** One **parent run** for the search, one **child run** per trial.
The parent keeps the winner; the children keep the evidence.

In [1]:
import sys, warnings
sys.path.insert(0, "../src")          # churn_data.py, churn_model.py
warnings.filterwarnings("ignore")

import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")   # the server you started in the terminal
mlflow.set_experiment("churn-workshop")
print("MLflow", mlflow.__version__)

MLflow 3.17.2


In [2]:
import optuna
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from churn_data import history, time_split
from churn_model import make_pipeline

optuna.logging.set_verbosity(optuna.logging.WARNING)
X_tr, y_tr, X_va, y_va = time_split(history())

## Step 1 · Define one trial = one child run

In [3]:
def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
    }
    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        model = make_pipeline(XGBClassifier(**params, eval_metric="logloss")).fit(X_tr, y_tr)
        auc = roc_auc_score(y_va, model.predict_proba(X_va)[:, 1])
        mlflow.log_params(params)
        mlflow.log_metric("val_auc", auc)
    return auc

## Step 2 · Run the search inside a parent run

In [4]:
with mlflow.start_run(run_name="xgb_optuna_search") as parent:
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=0))
    study.optimize(objective, n_trials=15)

    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.log_metric("best_val_auc", study.best_value)
    mlflow.set_tag("search", "optuna-tpe")

print("best AUC:", round(study.best_value, 4), study.best_params)

🏃 View run trial_0 at: http://127.0.0.1:5000/#/experiments/1/runs/f8b04836c40642f0aa9af3ec7becc964
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run trial_1 at: http://127.0.0.1:5000/#/experiments/1/runs/09799fa3a38d4d35b5351e84f8a12f23
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run trial_2 at: http://127.0.0.1:5000/#/experiments/1/runs/330a744171d740718d79b910b0724b32
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run trial_3 at: http://127.0.0.1:5000/#/experiments/1/runs/ff6768302dcf4a9eb05112ecf439a752
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run trial_4 at: http://127.0.0.1:5000/#/experiments/1/runs/c744bc392a404e6fac81de836aebc4d1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run trial_5 at: http://127.0.0.1:5000/#/experiments/1/runs/ba18c1735569453f90180fddc7f60817
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run trial_6 at: http://127.0.0.1:5000/#

👉 **In the UI:** expand the parent run to see its 15 children. Select them all → **Compare** →
*Parallel coordinates* chart. Which hyperparameter matters most?

## Step 3 · Find the best child from code
Child runs carry the tag `mlflow.parentRunId`. We'll reuse these parameters in Lab 3.

In [5]:
best = mlflow.search_runs(
    experiment_names=["churn-workshop"],
    filter_string=f"tags.mlflow.parentRunId = '{parent.info.run_id}'",
    order_by=["metrics.val_auc DESC"], max_results=1).iloc[0]
best_params = {k.replace("params.", ""): v for k, v in best.items() if k.startswith("params.")}
print(best["tags.mlflow.runName"], round(best["metrics.val_auc"], 4), best_params)

trial_6 0.7389 {'max_depth': '2', 'learning_rate': '0.01628359108363256', 'subsample': '0.9778675668198336', 'n_estimators': '400'}


## ✍️ Your turn (5 min)
Params come back from `search_runs` as **strings**. Convert `best_params` to the right types
(`int` for depth and estimators, `float` for the rest) so you can pass them to `XGBClassifier`.
Why do you think MLflow stores parameters as strings?

In [6]:
from xgboost import XGBClassifier

INT_PARAMS = {"max_depth", "n_estimators"}

typed_params = {
    k: int(v) if k in INT_PARAMS else float(v)
    for k, v in best_params.items()
}
print(typed_params)

xgb = XGBClassifier(**typed_params)

{'max_depth': 2, 'learning_rate': 0.01628359108363256, 'subsample': 0.9778675668198336, 'n_estimators': 400}


In [7]:
model = make_pipeline(xgb).fit(X_tr, y_tr)
print("val_auc:", roc_auc_score(y_va, model.predict_proba(X_va)[:, 1]))

val_auc: 0.7388953160811085


- **Universel** : un paramètre peut être un int, un float, un booléen, une liste, un nom d'algo, un chemin... Le texte accepte tout, quelle que soit la librairie (sklearn, XGBoost, PyTorch).
- **Stockage simple** : une seule colonne texte dans la base (SQLite, PostgreSQL), pas besoin de gérer un type par paramètre.